# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">9/28(월) 오전 · 예외처리 · 로깅 — 실습</mark>

오늘 오전의 도착점은 **깨진 줄이 섞인 로그 파일을 넣어도 멈추지 않고 끝까지 처리하는 파서**입니다. 4교시에 `log_parser.py` 로 저장합니다.

이 노트북은 혼자서 돌아갑니다. 지난 시간에 만든 파일이 없어도 됩니다. 아래 준비 셀이 오늘 쓸 데이터를 다시 만듭니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

### 0.1 맨 먼저 · 내 사본 만들기

1. 위 메뉴에서 파일 › 드라이브에 사본 저장을 누릅니다.
2. 제목이 「사본: …」으로 바뀌면 된 것입니다.
3. 사본을 만들지 않으면 내가 쓴 코드가 저장되지 않습니다.

### 0.2 셀 실행하기

셀을 누르고 Shift + Enter를 칩니다. 왼쪽 ▶를 눌러도 같습니다. **위에서부터 차례대로** 실행합니다.

### 0.3 오늘 오전의 순서

| 교시 | 무엇 |
|---|---|
| 2교시 | `try`·`except` — 에러가 나도 멈추지 않는다 |
| 3교시 | 예외 이름으로 나눠 잡기 · `finally` |
| 4교시 | `logging` · `log_parser.py` 조립 |

### 0.4 막혔을 때

1. 문제 바로 위의 문법 설명 셀을 다시 봅니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다. 왼쪽 목차(☰)에서 바로 갈 수 있습니다.
3. 정답 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다. 먼저 스스로 해 본 뒤 엽니다.

### 0.5 오늘 만든 코드는 어디에 남나

| | 무엇 |
|---|---|
| 코랩 | 연습장 — 창을 닫으면 여기 쓴 코드는 사라집니다 |
| 내 드라이브의 `agent_core` 폴더 | 작품 보관함 — 날마다 파일이 하나씩 쌓입니다 |

1. 그래서 마지막 실습은 코드를 `.py` 파일로 저장해 드라이브에 남깁니다.
2. 셀 맨 첫 줄의 `%%writefile 이름.py` 는 「이 셀을 실행하지 말고, 이 이름의 파일로 저장하라」는 뜻입니다. 실행 결과 대신 `Writing 이름.py` 가 나옵니다.
3. `!python 이름.py` 는 저장한 파일을 실행합니다. 앞의 `!` 는 「터미널 명령」이라는 표시입니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · 예외와 try·except</mark>


### <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 · 오늘 쓸 로그 파일 두 개</mark>

아래 두 셀을 먼저 실행합니다. 코랩 안에 파일 두 개가 생깁니다.

| 파일 | 무엇 |
|---|---|
| `sample_logs.csv` | 깔끔한 로그 17건 |
| `sample_logs_broken.csv` | 같은 17건에 **깨진 줄 5건**이 섞인 것 |

칸은 `시각,계정,사건,IP` 네 개이고 머리글 줄은 없습니다.


In [ ]:
%%writefile sample_logs.csv
09:01,kim01,LOGIN_OK,10.0.3.21
09:03,lee02,LOGIN_FAIL,10.0.7.5
09:05,admin,LOGIN_FAIL,10.0.9.8
09:07,park03,LOGIN_OK,10.0.4.11
09:09,kim01,LOGIN_OK,10.0.3.21
09:12,admin,LOGIN_FAIL,10.0.9.8
09:15,choi04,LOGIN_OK,10.0.6.2
09:18,lee02,LOGIN_FAIL,10.0.7.5
09:21,admin,LOGIN_FAIL,10.0.9.8
09:24,park03,LOGIN_OK,10.0.4.11
09:27,jung05,LOGIN_OK,10.0.8.30
09:30,kim01,LOGIN_OK,10.0.3.21
09:33,choi04,LOGIN_FAIL,10.0.6.2
09:36,jung05,LOGIN_OK,10.0.8.30
09:39,park03,LOGIN_OK,10.0.4.11
09:42,lee02,LOGIN_OK,10.0.7.5
09:45,kim01,LOGIN_OK,10.0.3.21


In [ ]:
%%writefile sample_logs_broken.csv
09:01,kim01,LOGIN_OK,10.0.3.21
09:03,lee02,LOGIN_FAIL,10.0.7.5
03:22,hacker
09:05,admin,LOGIN_FAIL,10.0.9.8
09:07,park03,LOGIN_OK,10.0.4.11
09:09,kim01,LOGIN_OK,10.0.3.21

09:12,admin,LOGIN_FAIL,10.0.9.8
09:15,choi04,LOGIN_OK,10.0.6.2
서버 점검 안내: 오늘 밤 2시부터
09:18,lee02,LOGIN_FAIL,10.0.7.5
09:21,admin,LOGIN_FAIL,10.0.9.8
09:24,park03,LOGIN_OK,10.0.4.11
09:27,jung05,LOGIN_OK,10.0.8.30
??:??,unknown,LOGIN_FAIL
09:30,kim01,LOGIN_OK,10.0.3.21
09:33,choi04,LOGIN_FAIL,10.0.6.2
09:36,jung05,LOGIN_OK,10.0.8.30
09:39,park03,LOGIN_OK,10.0.4.11
09:41,guest
09:42,lee02,LOGIN_OK,10.0.7.5
09:45,kim01,LOGIN_OK,10.0.3.21


In [ ]:
!ls  # 파일 두 개가 보이면 준비 끝입니다


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1 · try·except — 에러가 나도 멈추지 않는다</mark>


### 왜 필요한가

1. 지금까지 만든 코드는 로그가 전부 깔끔하다는 전제 위에 서 있습니다. 칸이 네 개씩 들어 있는 파일만 읽어 왔습니다.
2. 실제 로그에는 칸이 모자란 줄, 빈 줄, 로그인과 무관한 줄이 섞여 있습니다. 그런 줄을 만나면 파이썬은 그 자리에서 프로그램을 멈춥니다.
3. 로그 20만 줄 가운데 한 줄이 깨졌다고 나머지 19만 9,999줄을 못 읽으면 곤란합니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 예외 | 실행 도중 생긴 문제로 프로그램이 멈추는 것 |
| `IndexError` | 리스트에 없는 인덱스를 꺼냈을 때 나는 예외 |
| `try` | 「일단 시도해 볼 코드」를 담는 블록 |
| `except` | 예외가 났을 때 대신 실행할 코드를 담는 블록 |
| 트레이스백 | 예외가 난 자리까지의 경로를 보여 주는 빨간 글 |


### 쓰는 규칙 네 가지

1. `try:` 와 `except 예외이름:` 둘 다 줄 끝에 콜론을 붙이고, 안의 코드는 네 칸 들여씁니다.
2. `try` 안에서 예외가 나면 **남은 줄을 건너뛰고** 곧바로 `except` 로 갑니다.
3. 예외가 나지 않으면 `except` 는 실행되지 않습니다.
4. `except:` 만 쓰지 않고 **예외 이름을 지정**합니다. 이름을 안 쓰면 진짜 버그까지 가려집니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.1 `split()` 과 인덱스</mark>

| 말 | 뜻 |
|---|---|
| `split(",")` | 문자열을 쉼표에서 나눠 **리스트**로 돌려준다 |
| 인덱스 | 리스트에서 값의 자리를 가리키는 번호. **0부터** 센다 |
| `IndexError` | 리스트에 없는 인덱스를 꺼냈을 때 나는 예외 |

```python
text = "seoul,busan,daegu"
cities = text.split(",")
print(cities)        # ['seoul', 'busan', 'daegu']
print(cities[0])     # seoul
print(cities[2])     # daegu
```

값이 3개면 쓸 수 있는 인덱스는 **0·1·2** 까지입니다. `cities[3]` 은 `IndexError` 입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts)
```

막히면 바로 위 `1.1 split() 과 인덱스` 설명을 다시 봅니다.


In [ ]:
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts)


✅ `['09:01', 'kim01', 'LOGIN_OK']`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 리스트 전체가 아니라 한 자리만 꺼냅니다.

```python
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts[1])
```

막히면 바로 위 `1.1 split() 과 인덱스` 설명을 다시 봅니다.


In [ ]:
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts[1])


✅ `kim01`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-3 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 `parts[3]` 입니다. 값은 앞 문제와 똑같습니다.

```python
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts[3])
```

막히면 바로 위 `1.1 split() 과 인덱스` 설명을 다시 봅니다.


In [ ]:
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts[3])


✅ `IndexError: list index out of range`


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.2 `try` 와 `except`</mark>

```python
try:
    시도할 코드
except 예외이름:
    예외가 났을 때 실행할 코드
```

예외가 나면 `try` 의 남은 줄을 건너뛰고 `except` 로 갑니다. 프로그램은 멈추지 않고 계속 돌아갑니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.3 예시 · 예외가 났을 때</mark>

`int()` 는 숫자가 아닌 글자를 만나면 `ValueError` 로 멈춥니다. `try` 로 감싸면 멈추지 않습니다.


In [ ]:
try:
    n = int("abc")
    print(n)
except ValueError:
    print("숫자가 아닙니다")

print("프로그램은 계속 돌아갑니다")


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.4 예시 · 예외가 나지 않았을 때</mark>

같은 코드에 정상 값을 넣으면 `except` 는 실행되지 않습니다.


In [ ]:
try:
    n = int("123")
    print(n)
except ValueError:
    print("숫자가 아닙니다")


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.5 예시 · 반복문 안에서 건너뛰기</mark>

`for` 안에 `try` 를 두면 깨진 줄에서 `except` 로 갔다가 **다음 줄로 이어집니다.**


In [ ]:
lines = ["09:01,kim01,LOGIN_OK", "03:22,hacker", "09:05,lee02,LOGIN_FAIL"]

for line in lines:
    parts = line.split(",")
    try:
        print(parts[1], parts[2])
    except IndexError:
        print("깨진 줄 건너뜀:", line)

print("끝까지 읽었습니다")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-4 · 멈추지 않게 만들기</font></h3></td></tr></table>

주어진 코드는 `parts[3]` 에서 멈춥니다. 멈추지 않고 `칸이 모자랍니다` 를 출력하도록 고치시오.

1. `print(parts[3])` 을 `try:` 안으로 옮기고 네 칸 들여씁니다.
2. 그 아래에 `except IndexError:` 를 두고, 안에서 `칸이 모자랍니다` 를 출력합니다.
3. 맨 마지막 줄 `print("확인 끝")` 은 그대로 둡니다. 이 줄이 실행되면 성공입니다.

| | |
|---|---|
| 주어지는 값 | `line = "09:41,guest"` |
| 🎯 나와야 하는 결과 | `칸이 모자랍니다` · `확인 끝` |


In [ ]:
line = "09:41,guest"
parts = line.split(",")

print(parts[3])

print("확인 끝")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-5 · 깨진 파일을 끝까지 읽기</font></h3></td></tr></table>

`sample_logs_broken.csv` 를 한 줄씩 읽어 **계정 이름과 IP** 를 출력하시오. 깨진 줄에서 멈추면 안 됩니다.

1. `with open("sample_logs_broken.csv", encoding="utf-8") as f:` 로 파일을 엽니다.
2. `for line in f:` 로 한 줄씩 반복합니다.
3. 반복 안에서 `line.strip().split(",")` 으로 줄을 나눕니다.
4. 계정 이름과 IP 를 출력하는 줄을 `try:` 안에 둡니다. IP 는 마지막 칸입니다.
5. `except IndexError:` 에서는 아무것도 출력하지 않고 `pass` 만 씁니다.

| | |
|---|---|
| 주어지는 값 | 준비 셀에서 만든 `sample_logs_broken.csv` |
| 🎯 나와야 하는 결과 | 정상 17건의 계정·IP 가 출력되고 **끝까지 실행됨** |


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-6 · 깨진 줄이 몇 건인가</font></h3></td></tr></table>

같은 파일을 다시 읽으면서, **깨진 줄의 개수**를 세어 마지막에 한 번 출력하시오.

1. 반복을 시작하기 전에 숫자를 담을 이름을 하나 만들고 `0` 을 넣습니다.
2. `try` 안에서는 아무것도 출력하지 않고 `parts[3]` 을 꺼내기만 합니다.
3. `except IndexError:` 안에서 그 숫자를 1 늘립니다.
4. 반복이 끝난 뒤 그 숫자를 출력합니다.

| | |
|---|---|
| 주어지는 값 | `sample_logs_broken.csv` |
| 🎯 나와야 하는 결과 | `깨진 줄 5건` |


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>위 문제를 다 푼 사람만 풉니다. 처음 코딩하는 분은 건너뛰고 3교시로 가세요. 새 문법은 나오지 않습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 1-1 · 깨진 줄을 따로 모으기</font></h3></td></tr></table>

깨진 줄의 **개수**가 아니라 **내용**을 리스트에 모아 마지막에 출력하시오.

1. 반복을 시작하기 전에 빈 리스트를 하나 만듭니다.
2. `except IndexError:` 안에서 그 줄을 리스트에 `append` 합니다. 줄 끝의 줄바꿈은 `strip()` 으로 없앱니다.
3. 반복이 끝난 뒤 리스트를 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 깨진 줄 5개가 담긴 리스트 |


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 1-2 · 정상 줄만 딕셔너리로</font></h3></td></tr></table>

깨진 줄은 건너뛰고, **정상 줄만** 딕셔너리로 바꿔 리스트에 모으시오.

1. 칸 이름은 `time`·`user`·`event`·`ip` 네 개로 합니다.
2. 딕셔너리를 만드는 줄을 `try` 안에 둡니다.
3. 반복이 끝난 뒤 모은 개수를 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `정상 로그 17건` |


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 1-3 · 함수로 묶기</font></h3></td></tr></table>

줄 하나를 딕셔너리로 바꾸는 함수를 만들고, 그 함수를 `try` 안에서 부르시오.

1. 함수 이름은 `parse_line`, 매개변수는 `line` 하나로 합니다.
2. 함수 안에서 줄을 나누고 네 칸짜리 딕셔너리를 `return` 합니다.
3. 반복 안에서 `try` 로 감싸 부르고, 깨진 줄은 건너뜁니다.
4. 예외가 함수 **안**에서 나도 `except` 가 잡는다는 것을 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `정상 로그 17건` |


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `try:` | 일단 시도할 코드를 담는다 |
| `except IndexError:` | `IndexError` 가 났을 때 대신 실행한다 |
| `pass` | 아무것도 하지 않고 넘어간다 |
| 예외가 없으면 | `except` 는 실행되지 않는다 |
| 예외가 나면 | `try` 의 남은 줄을 건너뛰고 `except` 로 간다 |


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

각 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다. 정답 셀은 혼자서 실행됩니다.


In [ ]:
#@title 정답 1-4 { display-mode: "form" }
line = "09:41,guest"
parts = line.split(",")

try:
    print(parts[3])
except IndexError:
    print("칸이 모자랍니다")

print("확인 끝")


In [ ]:
#@title 정답 1-5 { display-mode: "form" }
with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            print(parts[1], parts[3])
        except IndexError:
            pass


In [ ]:
#@title 정답 1-6 { display-mode: "form" }
broken_count = 0

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            parts[3]
        except IndexError:
            broken_count = broken_count + 1

print(f"깨진 줄 {broken_count}건")


In [ ]:
#@title 정답 ⭐1-1 { display-mode: "form" }
broken_lines = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            parts[3]
        except IndexError:
            broken_lines.append(line.strip())

print(broken_lines)


In [ ]:
#@title 정답 ⭐1-2 { display-mode: "form" }
logs = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            logs.append({"time": parts[0], "user": parts[1], "event": parts[2], "ip": parts[3]})
        except IndexError:
            pass

print(f"정상 로그 {len(logs)}건")


In [ ]:
#@title 정답 ⭐1-3 { display-mode: "form" }
def parse_line(line):
    parts = line.strip().split(",")
    return {"time": parts[0], "user": parts[1], "event": parts[2], "ip": parts[3]}

logs = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        try:
            logs.append(parse_line(line))
        except IndexError:
            pass

print(f"정상 로그 {len(logs)}건")
